# Pandas

# 2. Analytical Brain of the Mission

While NumPy is the brute force for calculations, **Pandas** is our main terminal for data analysis and processing. It is a tool that allows us to take chaotic data from the field (geological samples, atmosphere records, crew logs) and transform it into structured information.

**Overview & Roadmap:**
* load data from external files (e.g., `.csv`, `.xlsx`)
* explore content and statistics (`describe`)
* clean data from transmission errors (`NaN`)
* filter samples according to conditions
* group, aggregate, and pivot data (e.g., average radiation per sector)
* export to files for saving / sending

## 2.0. Data Initialization (Simulation)

First, we must import the library. The standard alias shortcut for Pandas in the community is `pd`.

In [2]:
import pandas as pd

*We run this cell first. It will create a training file `geo_samples.csv` for us to work with.*

In [ ]:
import numpy as np

# simulation of data from the geological survey of Gale crater (Mars, position: 5.4°S 137.8°E)
data = {
    'sample_id': [f'S-{i:03d}' for i in range(1, 11)],
    'sector': ['Alpha', 'Alpha', 'Beta', 'Beta', 'Gamma', 'Alpha', 'Gamma', 'Beta', 'Gamma', 'Alpha'],
    'soil_type': ['Basalt', 'Clay', 'Sand', 'Basalt', 'Clay', np.nan, 'Sand', 'Basalt', 'Clay', 'Basalt'],
    'iron_content_pct': [15.2, 8.5, 12.1, 14.8, 9.2, 16.0, 11.5, np.nan, 8.9, 15.5],
    'water_ice_detected': [False, True, False, False, True, False, False, False, True, False],
    'radiation_level_msv': [0.5, 0.45, 1.2, 1.1, 0.6, 2.5, 0.55, 1.15, 0.62, 0.51]
}

df_sim = pd.DataFrame(data)

# Saving to CSV (external file simulation)
df_sim.to_csv('geo_samples.csv', index=False) # if we don't set the path otherwise, the file is saved to CWD (current working directory)

print("File 'geo_samples.csv' generated successfully in the lab.")

File 'geo_samples.csv' generated successfully in the lab.


## 2.1. Series vs. DataFrame

Pandas works with two basic data shapes. It is important to distinguish them.

1. `Series:` One-dimensional array (row, sequence) - for example, a drill sample (one column of data). It has an index (sampling depth) and values (analysis results).

2. `DataFrame:` Two-dimensional array (table). It is a collection of several **Series (columns)** aligned side by side - a complete geological report (rows = samples, columns = properties) on the area survey.

## 2.2. Loading and Exploration (Ingestion)

We load data from our rover and perform an initial inspection and analysis.

In [ ]:
# Loading data from a CSV file
df = pd.read_csv('geo_samples.csv')

# Quick analysis (look at data)
print("--- First 5 (HEAD) ---")
print(df.head()) # first 5 rows (records)
print("--- Last 5 (TAIL) ---")
print(df.tail()) # last 5 rows (records)

print("\n--- Structure (INFO) ---")
df.info() # number of records (rows), number of columns and their data type

print("\n--- Data Shape (SHAPE) ---")
print(f"Rows (samples): {df.shape[0]}, Columns (measurements): {df.shape[1]}")

print("\n--- Quick Statistics (DESCRIBE) ---")
print(df.describe()) # overview of min, max, average, and deviations - only for numerical data!

--- First 5 (HEAD) ---
  sample_id sector soil_type  iron_content_pct  water_ice_detected  \
0     S-001  Alpha    Basalt              15.2               False   
1     S-002  Alpha      Clay               8.5                True   
2     S-003   Beta      Sand              12.1               False   
3     S-004   Beta    Basalt              14.8               False   
4     S-005  Gamma      Clay               9.2                True   

   radiation_level_msv  
0                 0.50  
1                 0.45  
2                 1.20  
3                 1.10  
4                 0.60  
--- Last 5 (TAIL) ---
  sample_id sector soil_type  iron_content_pct  water_ice_detected  \
5     S-006  Alpha       NaN              16.0               False   
6     S-007  Gamma      Sand              11.5               False   
7     S-008   Beta    Basalt               NaN               False   
8     S-009  Gamma      Clay               8.9                True   
9     S-010  Alpha    Basalt      

In [ ]:
# loading from Excel (Legacy formats from Earth)
# Note: for working with Excel files, the 'openpyxl' library is needed (pip install openpyxl)

# df_excel = pd.read_excel('mission_supplies.xlsx')
# print(df_excel.info())

## 2.3. Data Cleaning (Decontamination)

In real operations, data is never perfect. Sensors fail, transmission drops out. Therefore, `NaN` (Not a Number) values may appear in our dataset, indicating **missing** or **corrupted** data.

In [ ]:
print(df.isna()) # verifies if NaN fields are present

# 1. if we don't know the soil type, the sample is useless 
# = we remove it from the dataset and create a new 'clean' dataframe, keeping original data
df_clean = df.dropna(subset=['soil_type']) # removal of all rows with a record if there is NaN in the 'soil_type' column

print(f"Original count: {len(df)}, count after 'drop': {len(df_clean)}") # check

# 2. we don't want to lose the sample and we want its other measured data 
# = we compensate for the error in one measurement by substituting our own value
avg_iron = df_clean['iron_content_pct'].mean() # prepare a "value" - e.g., average Fe content
df_clean = df_clean.copy() # create a copy for modifications
df_clean['iron_content_pct'] = df_clean['iron_content_pct'].fillna(avg_iron) 
# in DF, replace NaN fields in the given column with the average value

print(df_clean.isna()) # verify NaN fields

## 2.4. Selection and Filtration (Resource Hunting)

Often we are not interested in the entire dataset, but only **specific samples**. For example, we look for water or safe zones. We will create various `slices and selections` from the original DF. Caution - unless we purposefully make a **copy**, they are all in the form of a **view** - if we change something in them, we influence the original data too!

In [ ]:
# 1. column selection (Select)
radiation_report = df_clean[['sector', 'radiation_level_msv']] # we are interested only in sector and radiation
print("Radiation report:\n", radiation_report.head(3))

# 2. simple condition (Filter)
clay_samples = df_clean[df_clean['soil_type'] == 'Clay'] # only samples containing clay
print("\nClay samples:\n", clay_samples)

# 3. compound conditions (AND / OR)
# & = AND (simultaneously)
# | = OR (at least one)
valuable_samples = df_clean[
    (df_clean['iron_content_pct'] > 10) &
    (df_clean['water_ice_detected'] == True)
] # looking for samples that have high iron content (> 10%) AND water

print("\nPriority samples (Fe + H2O):\n", valuable_samples)

### 2.4.1. Working with Indices (loc vs iloc)

- `.loc:` We search by name (label).

- `.iloc:` We search by position (index, row number).

In [ ]:
# precise value localization (specific DF cell): 1st row (index 0), column 'soil_type'
print("First sample soil type:", df_clean.iloc[0, 2])

## 2.5 Manipulation and Transformation

We often have to modify obtained data for further calculations and subsequent use.

In [ ]:
# creating (and inserting) a new column = inserting a new column with "recalculated" values from another column into DF
df_clean['radiation_usv'] = df_clean['radiation_level_msv'] * 1000 # radiation conversion: mSv to µSv (x1000)

# sorting
# sort samples from highest radiation to lowest - according to the corresponding column
df_sorted = df_clean.sort_values(by='radiation_level_msv', ascending=False)

print("Critical values:\n", df_sorted[['sample_id', 'radiation_usv']].head(3))

## 2.6 Grouping, Aggregation, and Pivoting (Sector Analysis)
We need summary reports. How are individual survey sectors performing? Where is the most water?

1. `grouping:` divides DataFrame into groups (with the same values) in a given column
2. `aggregation:` with these groups, I can perform further operations - even in other columns (calculate average, max, min etc.)

Suitable for `summaries, analysis, and group metrics by key`. Grouping and aggregation allow analysis and combinations by `multiple categories` and detailed tracking of behavior of samples that share something in common.

3. `pivoting:` changing table arrangement - we need to select 3 columns:
    - **index** which column to use for rows / Y-axis
    - **columns** which column to use for columns / X-axis
    - **values** what data to use for values

The result is a more readable and understandable "table" - preparation for final visualization and presentation
usage: after grouping

In [ ]:
# grouping - by sector
sector_analysis = df_clean.groupby('sector')[['iron_content_pct', 'radiation_level_msv']].mean() # calculate average values - for each sector

# data from each sector becomes a separate group (grouping)
# in these "groups", we find average values for Fe content and radiation level (aggregation)
print("Sector Analysis:\n", sector_analysis)


# pivoting: "rearranged" table for visualization
# tracking average radiation by sector and soil type
pivot_map = df_clean.pivot_table(
    values='radiation_level_msv',
    index='sector',
    columns='soil_type',
    aggfunc='mean'
)
print("\nRadiation map (Pivot):\n", pivot_map)

## 2.7. Time Series and Trend (Atmosphere)

We load another data type – **continuous measurement** of the atmosphere (pressure and temperature) in time.

In [ ]:
# simulation of atmospheric data
time_data = {
    'timestamp': pd.date_range(start='2035-05-20 12:00', periods=10, freq='h'),
    'temperature_c': [-60, -62, -65, -68, -70, -65, -55, -50, -48, -52]
}
df_atmo = pd.DataFrame(time_data)

# Rolling Window
df_atmo['temp_smooth'] = df_atmo['temperature_c'].rolling(window=3).mean() 
# smoothen the temperature curve (average of 3 consecutive measurements)

print(df_atmo)

## 2.8. Data Export (Sending to Earth)

We must `save results of our analysis for further processing` or archiving.

In [ ]:
# export cleaned data to CSV

df_clean.to_csv('processed_geo_samples.csv', index=False)
# index=False = we don't save row numbers (0, 1, 2...)

print("Data exported and ready to be sent.")

In [ ]:
# export cleaned data to Excel
# Note: for writing to .xlsx, the 'openpyxl' library is needed (pip install openpyxl)

df_clean.to_excel('geo_report_final.xlsx', index=False, sheet_name='Geology_Data')
# index=False: we don't want to save row numbers
# sheet_name: naming a specific sheet in the workbook

print("Excel report saved and ready to be sent.")

## 2.9. Integration with API and Python Structures

We don't always work with data from files. We often communicate with `APIs` (drones, satellites) that send data as `JSON` (similar to Python dictionaries and lists). Pandas can work fluently with these formats in **both directions**.

#### Data Simulation from API (JSON -> DataFrame)

We received data from a meteorological probe in **JSON format**.

In [ ]:
# data from API (List of Dictionaries)
api_response = [
    {"sensor_id": "A-01", "pressure": 101.3, "location": "Crater Rim"},
    {"sensor_id": "B-02", "pressure": 98.2, "location": "Valley Floor"},
    {"sensor_id": "C-03", "pressure": 100.5, "location": "Base Camp"}
]

# conversion to DataFrame
df_api = pd.DataFrame(api_response)

print("Data from API probe:")
print(df_api)

#### Data Preparation for Sending (DataFrame -> Dict)

If we want to send processed data back to an API or another application, we must convert it from Pandas format back to native Python objects (dictionary).

In [ ]:
# export to dictionary (to_dict)
# orient='records' is the most common format for API (list of dictionaries [{column: value}, ...])
payload_for_api = df_api.to_dict(orient='records')

print("\nPayload ready (List of Dicts):")
print(payload_for_api)

## Practise - Training Simulations

### practise I: Deep Space Signal Analysis
We intercepted a series of weak signals from the Epsilon Eridani system. The data is currently in raw format (Python dictionary). We must convert it into a structured table and verify data integrity.

**Assignment:**

1. Create a DataFrame `df_signals` from incoming raw data.
    ```python
    raw_signals = {
        'source_id': ['AC-01', 'AC-02', 'AC-03', 'AC-04', 'AC-05', 'AC-06'],
        'frequency_mhz': [1420.4, 1420.5, None, 1421.0, 1420.4, 1420.8],
        'signal_strength': [0.95, 0.88, 0.02, 0.91, 0.99, 0.15],
        'is_confirmed': [True, True, False, True, True, False]
    }
    ```

2. Print the first 5 records for a quick data check.

3. Check the data types of columns (are frequencies numbers or text?) and table dimensions.

---

### practise II: Bio-signature Filtration on Europa
Underwater drones on Jupiter's moon Europa sent water analysis results. Some sensors failed due to pressure (missing data). We are looking for specific samples that might contain life.

**Assignment:**

1. Create a DataFrame `df_water` from incoming raw data.
    ```python
    data_europa = {
        'sample_id': range(1, 9),
        'depth_km': [5, 12, 8, 15, 2, 20, 11, 9],
        'ph_level': [6.5, 7.1, np.nan, 7.0, 6.8, 7.2, np.nan, 6.9],
        'temperature_c': [-2, -1.5, -2.1, np.nan, -1.8, -1.2, -2.0, np.nan],
        'organic_compounds': [0.1, 0.6, 0.0, 0.8, 0.05, 0.9, 0.2, 0.0]
    }
    ```

2. Sanitization: Remove all rows where `ph_level` data is missing (critical sensor error).

3. Correction: Replace missing values in `temperature_c` with the average temperature that we calculate from other measurements (imputation).

4. Search for life: Filter only those samples that have `amount of organic compounds higher than 0.5 AND AT THE SAME TIME were taken from depths greater than 10`.

---

### Practise III: Energy Balance of Mining Rovers
Our fleet of automated miners is working in various sectors of the asteroid belt. We need to find out which terrain type is the most energy-demanding to optimize batteries for the next generation of rovers.

**Assignment:**

1. Create a DataFrame `df_rovers` from incoming raw data.
    ```python
    rover_data = {
        'rover_id': ['R1', 'R2', 'R3', 'R4', 'R5', 'R6'],
        'terrain_type': ['Regolith', 'Rock', 'Ice', 'Regolith', 'Rock', 'Ice'],
        'energy_consumed_kwh': [45.2, 120.5, 80.1, 42.8, 115.3, 85.2],
        'mined_kg': [500, 300, 450, 520, 310, 440]
    }
    ```

2. Create a new column `efficiency`, calculated as the ratio of mined kg results versus energy consumed.

3. Group the data by terrain type.

4. Find the average energy consumption for individual terrain types and sort results from highest consumption to lowest.

5. Find the total amount of mined material for individual terrains.

## Homework (Project: Mars)
Probes and the first landing modules have collected critical data on the atmosphere and supplies in the vicinity of the planned base.

**Mission:** Analysis of living conditions and Ares base logistics.
**Goal:** Our task is to clean data, analyze environmental stability, and prepare a report for command.

0. Data Preparation (Simulation): Insert this code at the beginning and generate the input file `mars_env_log.csv`.

### Data generation for the last 100 sols (Martian days)
```python
dates = pd.date_range(start='2035-01-01', periods=100, freq='D')
mars_data = {
    'date': dates,
    'avg_temp_c': np.random.uniform(-80, -20, 100),
    'pressure_pa': np.random.uniform(600, 650, 100),
    'radiation_rem': np.random.uniform(0.1, 0.5, 100),
    'oxygen_tank_level': np.linspace(100, 60, 100) - np.random.uniform(0, 5, 100), # Declining supplies
    'solar_flare_detected': np.random.choice([True, False], 100, p=[0.1, 0.9])
}

df_mars = pd.DataFrame(mars_data)

# Inserting sensor errors (Damage simulation)
df_mars.loc[10:15, 'avg_temp_c'] = np.nan
df_mars.loc[50:52, 'pressure_pa'] = np.nan

# Saving the file
df_mars.to_csv('mars_env_log.csv', index=False)
print("Mars environment log downloaded.")
```

### Assignment:

1. Loading and Inspection:

    - Load the file `mars_env_log.csv`.

    - Convert the `date` column to datetime objects (using pd.to_datetime).

    - Check how many records contain NaN (sensor outages).

2. Decontamination (Cleaning):

    - Replace missing `pressure_pa data` with the average pressure value for the entire period.

    - Missing `avg_temp_c` data is considered a critical sensor error. Completely remove rows with missing temperature from the dataset.

3. Trend Analysis (Rolling):

    - Create a new column `radiation_smooth`, which will represent a 7-day rolling average of radiation. This will help us filter out daily fluctuations and see the trend.

4. Safety Protocol (Filtering):

    - Filter days when a solar flare was detected AND AT THE SAME TIME oxygen level dropped below 70%.

    - Save these critical days into a new DataFrame `critical_days`.

5. Final Report (Export):

    - Export the final cleaned DataFrame (with the new radiation column) to Excel under the name `mars_report_phase1.xlsx`.

    - Export critical days (`critical_days`) to a separate CSV `alerts.csv` for the security team.

---
#### © Jiří Svoboda (George Freedom)
- Web: https://GeorgeFreedom.com
- LinkedIn: https://www.linkedin.com/in/georgefreedom/
- Let's talk: https://cal.com/georgefreedom